# Preparación de datos — Selector de Materiales JARVIS

**Etapa 3 del proyecto · Ingeniería de Materiales · Dataset 5 (JARVIS-DFT)**

Este notebook toma el `Dataset_5.csv` original, repite la limpieza de la Etapa 1, agrega las variables
de la Etapa 2 (familia química, K/G, residuo K–G) y calcula un **indicador de confiabilidad** por material.
El resultado se exporta a `datos.js`, que es el archivo que lee la página `index.html`.

Flujo: `Dataset_5.csv` → este notebook → `datos.js` → `index.html` (GitHub Pages)

## 1. Carga

En Colab se monta Drive; fuera de Colab el CSV se busca junto al notebook (por ejemplo, dentro del repositorio).

In [1]:
import re, json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA = '/content/drive/MyDrive/Colab Notebooks/Dataset_5.csv'
except ImportError:
    RUTA = 'Dataset_5.csv'

df = pd.read_csv(RUTA)
print('Dataset original:', df.shape)

Dataset original: (3000, 22)


## 2. Limpieza (criterios de la Etapa 1)

`'na'` se convierte en `NaN` y las 11 columnas de propiedades pasan a numéricas. No se elimina ninguna fila:
los problemas se marcan para que la herramienta los muestre al usuario.

In [2]:
cols_num = ['mbj_bandgap', 'bulk_modulus_kv', 'shear_modulus_gv', 'poisson', 'epsx', 'epsy', 'epsz',
            'magmom_oszicar', 'dfpt_piezo_max_eij', 'dfpt_piezo_max_dij', 'Tc_supercon']
d = df.replace('na', np.nan)
for c in cols_num:
    d[c] = pd.to_numeric(d[c], errors='raise')

# Posibles registros redundantes (misma fórmula, grupo espacial y nat con propiedades casi iguales)
cols_est = ['formula', 'spg_number', 'nat']
g = d[d.duplicated(cols_est, keep=False)].groupby(cols_est)[['density', 'formation_energy_peratom', 'ehull']]
r = g.agg(lambda s: s.max() - s.min())
claves = set(r.index[(r['density'] <= 0.05) & (r['formation_energy_peratom'] <= 0.01) & (r['ehull'] <= 0.01)])
d['posible_redundante'] = [tuple(x) in claves for x in d[cols_est].values]

print('Faltantes totales:', int(d[cols_num + ['dimensionality']].isna().sum().sum()))
print('Materiales con K y G:', int(d['bulk_modulus_kv'].notna().sum()))
print('Posibles redundantes:', int(d['posible_redundante'].sum()))

Faltantes totales: 20185
Materiales con K y G: 1050
Posibles redundantes: 13


## 3. Variables derivadas

- **Familia química** (misma regla jerárquica de la Etapa 2).
- **Propiedades elásticas derivadas** de los promedios de Voigt K y G:

$$E = \frac{9KG}{3K+G}, \qquad \nu_V = \frac{3K-2G}{2(3K+G)}, \qquad \text{Pugh} = K/G$$

- **Rigidez específica** $E/\rho$ (GPa·cm³/g), útil para seleccionar materiales ligeros y rígidos.

In [3]:
NOBLES = {'He', 'Ne', 'Ar', 'Kr', 'Xe', 'Rn'}
HAL = {'F', 'Cl', 'Br', 'I'}; CALC = {'S', 'Se', 'Te'}; BCN = {'B', 'C', 'N'}; SIP = {'Si', 'Ge', 'P', 'As', 'Sb'}

def elementos(f):
    return set(re.findall(r'[A-Z][a-z]?', f))

def familia(f):
    e = elementos(f)
    if e & NOBLES: return 'Con gas noble'
    if 'O' in e:   return 'Óxidos'
    if e & HAL:    return 'Halogenuros'
    if e & CALC:   return 'Calcogenuros'
    if e & BCN:    return 'Boruros, carburos, nitruros'
    if 'H' in e:   return 'Hidruros'
    if e & SIP:    return 'Siliciuros y pnicturos'
    return 'Intermetálicos'

d['familia'] = d['formula'].apply(familia)
d['elementos'] = d['formula'].apply(lambda f: sorted(elementos(f)))

K, G = d['bulk_modulus_kv'], d['shear_modulus_gv']
pos = (K > 0) & (G > 0)
d['E'] = np.where(pos, 9 * K * G / (3 * K + G), np.nan)
d['nu_V'] = np.where(pos, (3 * K - 2 * G) / (2 * (3 * K + G)), np.nan)
d['K_G'] = np.where(pos, K / G, np.nan)
d['E_rho'] = d['E'] / d['density']
print(d['familia'].value_counts().to_string())
print('\nMateriales con K > 0 y G > 0:', int(pos.sum()))

familia
Intermetálicos                 908
Óxidos                         823
Halogenuros                    344
Siliciuros y pnicturos         340
Calcogenuros                   331
Boruros, carburos, nitruros    193
Hidruros                        38
Con gas noble                   23

Materiales con K > 0 y G > 0: 951


## 4. Residuo respecto a la relación K–G (Etapa 2)

Se ajusta el modelo M1 de la Etapa 2, $\log_{10}G = a_{\text{familia}} + b\,\log_{10}K$, en las seis familias con
al menos 30 materiales. El residuo se expresa como **z robusto** (mediana y MAD). Un |z| > 3.5 indica que el G del
material es muy distinto de lo esperado para su K y su familia.

In [4]:
FAMILIAS = ['Intermetálicos', 'Siliciuros y pnicturos', 'Boruros, carburos, nitruros',
            'Óxidos', 'Halogenuros', 'Calcogenuros']
m = d[pos & d['familia'].isin(FAMILIAS)]
X = np.column_stack([pd.get_dummies(m['familia'])[FAMILIAS].astype(float).values, np.log10(m['bulk_modulus_kv'])])
y = np.log10(m['shear_modulus_gv']).values
beta = np.linalg.lstsq(X, y, rcond=None)[0]
res = y - X @ beta
mad = 1.4826 * np.median(np.abs(res - np.median(res)))
d['z_KG'] = np.nan
d.loc[m.index, 'z_KG'] = (res - np.median(res)) / mad

print('Muestra del ajuste:', len(m), '| pendiente común:', round(beta[-1], 3))
print('Atípicos |z| > 3.5:', int((d['z_KG'].abs() > 3.5).sum()))

Muestra del ajuste: 932 | pendiente común: 0.935
Atípicos |z| > 3.5: 39


## 5. Indicadores de confiabilidad y de estabilidad

Los datos son **cálculos DFT**, no mediciones. La herramienta no oculta materiales dudosos: los muestra con avisos.
Se usan **dos indicadores separados**, porque responden preguntas distintas:

- **Confiabilidad del dato:** ¿el cálculo parece coherente? Cada aviso tiene un peso y la suma define el nivel.
- **Estabilidad termodinámica:** ¿es probable que el material exista? Se basa solo en ehull.

En la Etapa 1 se vio que el 78.5 % del dataset tiene ehull > 0.5 eV/átomo. Si el ehull formara parte de la confiabilidad,
casi todos los materiales quedarían en el mismo nivel y el indicador no ayudaría a distinguirlos.

| Aviso | Condición | Peso |
|---|---|---|
| Módulo no positivo | K ≤ 0 o G ≤ 0 | crítico |
| Contiene gas noble | He, Ne, Ar, Kr, Xe, Rn | crítico |
| Cizalla muy baja | 0 < G < 5 GPa | 1 |
| Atípico K–G | \|z\| > 3.5 | 1 |
| Poisson reportado incoherente | fuera de [−1, 0.5] | 0.5 |
| Baja dimensionalidad | no es 3D-bulk | 0.5 |
| Densidad < 1 g/cm³ | celda con vacío | 1 |
| ehull < Ef | inconsistencia termodinámica | 1 |
| Posible registro redundante | Etapa 1 | 0.5 |

**Nivel de confiabilidad:** *Baja* si hay un aviso crítico o la suma es ≥ 1.5; *Media* si la suma es 0.5 o 1; *Alta* si no hay avisos.

**Estabilidad:** *Estable* si ehull ≤ 0.1 eV/átomo; *Metaestable* si 0.1 < ehull ≤ 0.5; *Hipotético* si ehull > 0.5.

In [5]:
AVISOS = [
    ('modulo_no_positivo', 'Módulo K o G no positivo', 'critico', ((K <= 0) | (G <= 0)).fillna(False)),
    ('gas_noble', 'Contiene un gas noble', 'critico', d['elementos'].apply(lambda e: bool(set(e) & NOBLES))),
    ('cizalla_baja', 'G < 5 GPa: muy sensible al error de cálculo', 1, ((G > 0) & (G < 5)).fillna(False)),
    ('atipico_KG', 'G muy distinto de lo esperado para su K y familia', 1, (d['z_KG'].abs() > 3.5).fillna(False)),
    ('poisson', 'Poisson reportado fuera de [−1, 0.5]', 0.5, ((d['poisson'] < -1) | (d['poisson'] > 0.5)).fillna(False)),
    ('baja_dim', 'No es 3D-bulk: las propiedades dependen del modelo de celda', 0.5, d['dimensionality'] != '3D-bulk'),
    ('densidad_baja', 'Densidad < 1 g/cm³: la celda incluye vacío', 1, d['density'] < 1),
    ('ehull_menor_Ef', 'ehull < energía de formación: inconsistencia termodinámica', 1,
     d['ehull'] < d['formation_energy_peratom'] - 0.001),
    ('redundante', 'Posible registro redundante', 0.5, d['posible_redundante']),
]
d['avisos'] = [[] for _ in range(len(d))]
puntaje = pd.Series(0.0, index=d.index); critico = pd.Series(False, index=d.index)
for clave, texto, peso, mask in AVISOS:
    mask = mask.fillna(False).astype(bool)
    for i in d.index[mask]:
        d.at[i, 'avisos'].append(clave)
    if peso == 'critico':
        critico |= mask
    else:
        puntaje += mask * peso
d['confiabilidad'] = np.select([critico | (puntaje >= 1.5), puntaje > 0], ['Baja', 'Media'], 'Alta')
d['estabilidad'] = np.select([d['ehull'] <= 0.1, d['ehull'] <= 0.5], ['Estable', 'Metaestable'], 'Hipotético')

print('Todos los materiales:')
print(d['confiabilidad'].value_counts().reindex(['Alta', 'Media', 'Baja']).to_string())
print('\nMateriales con K y G:')
print(d.loc[d['bulk_modulus_kv'].notna(), 'confiabilidad'].value_counts().reindex(['Alta', 'Media', 'Baja']).to_string())
print('\nEstabilidad:')
print(d['estabilidad'].value_counts().to_string())
print('\nFrecuencia de cada aviso:')
print(pd.Series({t: int(mk.fillna(False).sum()) for _, t, _, mk in AVISOS}).to_string())

Todos los materiales:
confiabilidad
Alta     1434
Media    1336
Baja      230

Materiales con K y G:
confiabilidad
Alta     431
Media    447
Baja     172

Estabilidad:
estabilidad
Hipotético     2355
Estable         346
Metaestable     299

Frecuencia de cada aviso:
Módulo K o G no positivo                                         99
Contiene un gas noble                                            23
G < 5 GPa: muy sensible al error de cálculo                      57
G muy distinto de lo esperado para su K y familia                39
Poisson reportado fuera de [−1, 0.5]                            135
No es 3D-bulk: las propiedades dependen del modelo de celda    1479
Densidad < 1 g/cm³: la celda incluye vacío                       26
ehull < energía de formación: inconsistencia termodinámica       60
Posible registro redundante                                      13


## 6. Exportación para la página

Se exportan las columnas que usa la herramienta, con nombres cortos. Los `NaN` se convierten en `null`.
El archivo `datos.js` define una variable global `DATOS`; así la página funciona tanto en GitHub Pages
como al abrir `index.html` directamente desde la computadora.

In [6]:
salida = pd.DataFrame({
    'jid': d['jid'], 'f': d['formula'], 'fam': d['familia'], 'crys': d['crys'], 'spg': d['spg_symbol'],
    'dim': d['dimensionality'].fillna('sin dato'), 'nat': d['nat'],
    'rho': d['density'], 'Ef': d['formation_energy_peratom'], 'eh': d['ehull'],
    'gap': d['optb88vdw_bandgap'], 'mbj': d['mbj_bandgap'], 'mag': d['magmom_oszicar'],
    'K': d['bulk_modulus_kv'], 'G': d['shear_modulus_gv'], 'nuR': d['poisson'],
    'E': d['E'].round(2), 'nu': d['nu_V'].round(3), 'KG': d['K_G'].round(3), 'Er': d['E_rho'].round(2),
    'z': d['z_KG'].round(2), 'conf': d['confiabilidad'], 'est': d['estabilidad'], 'av': d['avisos'],
})
registros = json.loads(salida.to_json(orient='records', force_ascii=False))
meta = {
    'fuente': 'JARVIS-DFT (NIST), Dataset 5 del curso',
    'n': len(registros),
    'pendiente_KG': round(float(beta[-1]), 4),
    'intercepto_KG': {f: round(float(b), 4) for f, b in zip(FAMILIAS, beta[:-1])},
    'mad_KG': round(float(mad), 4),
    'avisos': {clave: {'texto': texto, 'peso': peso} for clave, texto, peso, _ in AVISOS},
}
with open('datos.js', 'w', encoding='utf-8') as fh:
    fh.write('// Generado por preparar_datos.ipynb. No editar a mano.\n')
    fh.write('const META = ' + json.dumps(meta, ensure_ascii=False) + ';\n')
    fh.write('const DATOS = ' + json.dumps(registros, ensure_ascii=False, separators=(',', ':')) + ';\n')

import os
print('Registros exportados:', len(registros))
print('Tamaño de datos.js:', round(os.path.getsize('datos.js') / 1024), 'KB')
print(registros[3])

Registros exportados: 3000
Tamaño de datos.js: 928 KB
{'jid': 'JVASP-37036', 'f': 'Ti3In', 'fam': 'Intermetálicos', 'crys': 'cubic', 'spg': 'Fm-3m', 'dim': '3D-bulk', 'nat': 4, 'rho': 5.949, 'Ef': -0.01939, 'eh': 2.7393, 'gap': 0.0, 'mbj': 0.0, 'mag': 1.725, 'K': 106.7, 'G': 25.2, 'nuR': 0.43, 'E': 70.08, 'nu': 0.391, 'KG': 4.234, 'Er': 11.78, 'z': -1.53, 'conf': 'Alta', 'est': 'Hipotético', 'av': []}


## 7. Descarga del archivo (solo en Colab)

En Colab, la siguiente celda descarga `datos.js` para subirlo al repositorio junto a `index.html`.

In [7]:
try:
    from google.colab import files
    files.download('datos.js')
except ImportError:
    print('Fuera de Colab: datos.js quedó en la carpeta actual.')

Fuera de Colab: datos.js quedó en la carpeta actual.
